# Phase 2: Full FT CaseHOLD + Random Label Baseline

AutoDL H800 80GB — 补充实验

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E9  | Full FT CaseHOLD Qwen  | Full FT | CaseHOLD | Qwen2.5-1.5B |
| E10 | Full FT CaseHOLD Llama | Full FT | CaseHOLD | Llama-3.2-1B |
| E11 | Random BillSum Qwen    | LoRA (shuffled labels) | BillSum  | Qwen2.5-1.5B |
| E12 | Random BillSum Llama   | LoRA (shuffled labels) | BillSum  | Llama-3.2-1B |
| E13 | Random CaseHOLD Qwen   | LoRA (shuffled labels) | CaseHOLD | Qwen2.5-1.5B |
| E14 | Random CaseHOLD Llama  | LoRA (shuffled labels) | CaseHOLD | Llama-3.2-1B |

## 0. 环境准备

In [ ]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)

# 软链到数据盘
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

In [ ]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub
print('\n✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel')

In [ ]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# HuggingFace 登录 (Llama 需要)
from huggingface_hub import login
login()
print('HuggingFace 登录成功')

## 1. 数据验证

In [ ]:
import os
files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]
for f in files:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else -1
    status = f'✓ {size} KB' if size >= 0 else '✗ 缺失！'
    print(f'{status}  {f}')

---
## 2. Full Fine-Tuning — CaseHOLD

使用 `train.py`（无 `lora` 节时自动全量微调）。

### E9: Full FT — CaseHOLD × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/full_casehold_qwen.yaml 2>&1 | tee logs/full_casehold_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_casehold_qwen.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/full_casehold_qwen/predictions_test.jsonl --output results/casehold/full_qwen_test.json
print('评估完成')
!cat results/casehold/full_qwen_test.json

### E10: Full FT — CaseHOLD × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_casehold_llama.yaml 2>&1 | tee logs/full_casehold_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_casehold_llama.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/full_casehold_llama/predictions_test.jsonl --output results/casehold/full_llama_test.json
print('评估完成')
!cat results/casehold/full_llama_test.json

---
## 3. Random Label Baseline (LoRA)

将训练集的 output 打乱（input 不变），使 input→output 映射完全随机。  
如果模型确实学到了任务知识，random label 训练后在真实测试集上的表现应大幅低于正常训练。

### 3.0 生成 Random Label 数据

In [ ]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label Datasets ===')
shuffle_labels('data/billsum/train_sft.jsonl',       'data/billsum/train_sft_random.jsonl')
shuffle_labels('data/billsum/val_sft.jsonl',         'data/billsum/val_sft_random.jsonl')
shuffle_labels('data/casehold/train_mc.jsonl',       'data/casehold/train_mc_random.jsonl')
shuffle_labels('data/casehold/validation_mc.jsonl',  'data/casehold/validation_mc_random.jsonl')
print('\nDone. Test files are NOT shuffled (evaluate on real data).')

In [ ]:
# 验证 mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (BillSum ~1.0, CaseHOLD ~0.8):')
print(f'  BillSum train:  {mismatch_rate("data/billsum/train_sft.jsonl", "data/billsum/train_sft_random.jsonl"):.4f}')
print(f'  BillSum val:    {mismatch_rate("data/billsum/val_sft.jsonl", "data/billsum/val_sft_random.jsonl"):.4f}')
print(f'  CaseHOLD train: {mismatch_rate("data/casehold/train_mc.jsonl", "data/casehold/train_mc_random.jsonl"):.4f}')
print(f'  CaseHOLD val:   {mismatch_rate("data/casehold/validation_mc.jsonl", "data/casehold/validation_mc_random.jsonl"):.4f}')

### E11: Random Label LoRA — BillSum × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_billsum_qwen.yaml 2>&1 | tee logs/random_billsum_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_qwen.yaml --split test_us --batch_size 8
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_qwen.yaml --split test_ca --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_qwen/predictions_test_us.jsonl --output results/billsum/random_qwen_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_qwen/predictions_test_ca.jsonl --output results/billsum/random_qwen_test_ca.json
print('评估完成')
!cat results/billsum/random_qwen_test_us.json

### E12: Random Label LoRA — BillSum × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_billsum_llama.yaml 2>&1 | tee logs/random_billsum_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_llama.yaml --split test_us --batch_size 8
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_llama.yaml --split test_ca --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_llama/predictions_test_us.jsonl --output results/billsum/random_llama_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_llama/predictions_test_ca.jsonl --output results/billsum/random_llama_test_ca.json
print('评估完成')
!cat results/billsum/random_llama_test_us.json

### E13: Random Label LoRA — CaseHOLD × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_casehold_qwen.yaml 2>&1 | tee logs/random_casehold_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_casehold_qwen.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/random_casehold_qwen/predictions_test.jsonl --output results/casehold/random_qwen_test.json
print('评估完成')
!cat results/casehold/random_qwen_test.json

### E14: Random Label LoRA — CaseHOLD × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_casehold_llama.yaml 2>&1 | tee logs/random_casehold_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_casehold_llama.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/random_casehold_llama/predictions_test.jsonl --output results/casehold/random_llama_test.json
print('评估完成')
!cat results/casehold/random_llama_test.json

---
## 4. 汇总所有结果

In [ ]:
import json, os

results = [
    ('full_casehold_qwen',    'results/casehold/full_qwen_test.json'),
    ('full_casehold_llama',   'results/casehold/full_llama_test.json'),
    ('random_billsum_qwen  (US)', 'results/billsum/random_qwen_test_us.json'),
    ('random_billsum_qwen  (CA)', 'results/billsum/random_qwen_test_ca.json'),
    ('random_billsum_llama (US)', 'results/billsum/random_llama_test_us.json'),
    ('random_billsum_llama (CA)', 'results/billsum/random_llama_test_ca.json'),
    ('random_casehold_qwen',  'results/casehold/random_qwen_test.json'),
    ('random_casehold_llama', 'results/casehold/random_llama_test.json'),
]

print(f'{"实验":<30} {"指标":<15} {"值":>8}')
print('-' * 55)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<30} {"未完成":<15}')
        continue
    with open(path) as f:
        d = json.load(f)
    if 'rouge2' in d:
        val = d['rouge2']['mean'] if isinstance(d['rouge2'], dict) else d['rouge2']
        print(f'{name:<30} {"rouge2":<15} {val:>8.4f}')
    elif 'accuracy' in d:
        val = d['accuracy'] if isinstance(d['accuracy'], (int, float)) else d['accuracy']['mean']
        print(f'{name:<30} {"accuracy":<15} {val:>8.4f}')

## 5. 提交结果

In [ ]:
!cd /root/MLP && git add results/ && git commit -m 'results: add Full FT CaseHOLD + random label baseline' && git push origin main